# 🔬 AI-Powered Drug Repurposing Pipeline

**Objective:** Identify candidate drugs to treat diseases by matching drug compounds against disease-associated protein targets using DeepPurpose deep learning models.

## Pipeline Overview:
1. **Disease-to-Targets**: Query Open Targets API to find proteins associated with a disease
2. **Protein Sequences**: Fetch amino acid sequences from UniProt
3. **Drug Library**: Load FDA-approved drugs with SMILES from TDC
4. **AI Screening**: Use DeepPurpose MPNN_CNN model to predict binding affinities
5. **Result Processing**: Rank and classify potential drug candidates

---

**Note:** This notebook requires internet access for API calls and will download 600+ FDA-approved drugs from TDC.

In [ ]:
# Install required dependencies (run once)
# Uncomment and run these if libraries are not installed

# !pip install DeepPurpose
# !pip install pandas
# !pip install requests
# !pip install git+https://github.com/Alantic/TDC.git
# !pip install rdkit
# !pip install git+https://github.com/bp-kelley/descriptastorus
# !pip install pandas-flavor

print("✅ Dependencies installation cell (see comments for commands to run if needed)")

In [ ]:
!pip install PyTDC

## Section 1: Import Required Libraries

In [8]:
# Phase 0: Setup and Imports
import requests
import pandas as pd
from DeepPurpose import DTI as models
from DeepPurpose.utils import data_process
from tdc.single_pred import ADME

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


## Section 2: Disease-to-Targets Mapping Using Open Targets API

In [9]:
def get_disease_targets(disease_name):
    """
    Fetches the top 10 target proteins associated with a specific disease.
    Uses the Open Targets GraphQL API.
    """
    url = "https://api.platform.opentargets.org/api/v4/graphql"

    # Step 1.1: Search for the Disease ID (EFO ID)
    search_query = """
    query searchDisease($queryString: String!) {
      search(queryString: $queryString, entityNames: ["disease"]) {
        hits { id name }
      }
    }
    """
    variables = {"queryString": disease_name}
    response = requests.post(url, json={'query': search_query, 'variables': variables})
    search_results = response.json().get('data', {}).get('search', {}).get('hits', [])

    if not search_results:
        return None

    disease_id = search_results[0]['id']
    print(f"✅ Target Disease Identified: {search_results[0]['name']} ({disease_id})")

    # Step 1.2: Fetch top 10 associated proteins
    target_query = """
    query diseaseTargets($diseaseId: String!) {
      disease(efoId: $diseaseId) {
        associatedTargets(page: {index: 0, size: 10}) {
          rows {
            target { approvedSymbol approvedName }
            score
          }
        }
      }
    }
    """
    variables = {"diseaseId": disease_id}
    response = requests.post(url, json={'query': target_query, 'variables': variables})
    targets_data = response.json().get('data', {}).get('disease', {}).get('associatedTargets', {}).get('rows', [])

    return [{"symbol": r['target']['approvedSymbol'], "score": r['score']} for r in targets_data]

print("✅ Disease mapping function defined")

✅ Disease mapping function defined


## Section 3: Fetch Protein Sequences from UniProt

In [10]:
def get_protein_sequences(targets_list):
    """
    Retrieves Amino Acid sequences for the identified proteins.
    Uses the UniProt REST API.
    """
    final_targets = []
    print("\n🧬 Fetching Amino Acid Sequences from UniProt...")

    for t in targets_list:
        symbol = t['symbol']
        url = f"https://rest.uniprot.org/uniprotkb/search?query={symbol}&format=json"
        try:
            response = requests.get(url).json()
            sequence = response['results'][0]['sequence']['value']
            t['sequence'] = sequence
            final_targets.append(t)
            print(f"   ✅ Successfully fetched sequence for: {symbol}")
        except Exception as e:
            print(f"   ⚠️ Failed to fetch sequence for: {symbol} ({str(e)[:50]})")

    print(f"\n✅ Successfully fetched sequences for {len(final_targets)}/{len(targets_list)} targets")
    return final_targets

print("✅ Protein sequence function defined")

✅ Protein sequence function defined


## Section 4: Load FDA-Approved Drug Library from TDC

In [11]:
def load_drug_library(dataset_name='Half_Life_Obach', limit=None):
    """
    Loads FDA-approved drugs with their SMILES strings using TDC.
    
    Args:
        dataset_name: TDC dataset name (default: 'Half_Life_Obach')
        limit: Max number of drugs to load (default: None = load all)
    """
    print(f"\n💊 Loading FDA-Approved Drug Library from TDC ({dataset_name})...")
    
    try:
        data = ADME(name=dataset_name).get_data()
        print(f"   Downloaded {len(data)} total drugs from TDC")
        
        library = []
        for idx, row in data.iterrows():
            smiles = row.get('Drug', '')
            
            # Skip invalid SMILES
            if not smiles or len(str(smiles).strip()) == 0:
                continue
                
            drug_id = idx + 1
            library.append({
                "name": f"Drug_{drug_id}",
                "smiles": str(smiles).strip(),
                "drug_id": str(drug_id),
                "source": "TDC"
            })
            
            if limit and len(library) >= limit:
                break
        
        print(f"✅ Loaded {len(library)} valid drugs ready for screening.")
        print(f"   Sample SMILES: {library[0]['smiles'][:60]}...")
        return library
        
    except Exception as e:
        print(f"❌ Error loading TDC: {str(e)}")
        raise

print("✅ Drug library function defined")

✅ Drug library function defined


## Section 5: AI Virtual Screening with DeepPurpose

Using DeepPurpose MPNN_CNN_BindingDB model with:
- **MPNN** (Message Passing Neural Network) for drug/compound encoding
- **CNN** (Convolutional Neural Network) for protein sequence encoding

In [12]:
def run_virtual_screening(drug_lib, target_lib, ai_model):
    """
    Performs the AI-based matching between all drugs and targets.
    Uses MPNN for drug encoding and CNN for protein encoding.
    
    Args:
        drug_lib: List of drugs with 'name' and 'smiles'
        target_lib: List of targets with 'symbol' and 'sequence'
        ai_model: Pre-trained DeepPurpose model
        
    Returns:
        List of predictions with drug_name, target_symbol, and score
    """
    X_drugs = []
    X_targets = []
    pair_info = []

    print("\n🚀 Preparing data for AI Prediction (Virtual Screening)...")
    
    # Create all drug-target pairs
    for drug in drug_lib:
        for target in target_lib:
            X_drugs.append(drug['smiles'])
            X_targets.append(target['sequence'])
            pair_info.append({
                'drug_name': drug['name'],
                'target_symbol': target['symbol']
            })

    print(f"   Total drug-target pairs: {len(X_drugs)}")
    print(f"   Encoding compounds (MPNN) and proteins (CNN)...")
    
    # Encode data for DeepPurpose
    X_pred = data_process(
        X_drug=X_drugs,
        X_target=X_targets,
        y=[0]*len(X_drugs),
        drug_encoding='MPNN',        # Message Passing Neural Network
        target_encoding='CNN',        # Convolutional Neural Network
        split_method='no_split'
    )

    print("🧠 AI is predicting Binding Affinities...")
    scores = ai_model.predict(X_pred)

    # Combine results
    results = []
    for i, score in enumerate(scores):
        results.append({
            "drug_name": pair_info[i]['drug_name'],
            "target_symbol": pair_info[i]['target_symbol'],
            "score": round(float(score), 4)
        })
    
    print(f"✅ Completed {len(results)} predictions")
    return results

print("✅ AI screening function defined")

✅ AI screening function defined


## Section 6: Process and Filter Results

In [13]:
def process_final_results(results, known_drugs):
    """
    Sorts results and labels them as 'Known Treatment' or 'Potential Discovery'.
    
    Args:
        results: List of prediction results
        known_drugs: List of known drug names for the disease
        
    Returns:
        Sorted and classified results
    """
    sorted_results = sorted(results, key=lambda x: x['score'], reverse=True)
    final_output = []

    for res in sorted_results:
        # Check if drug is already known for this disease
        is_known = any(known.lower() in res['drug_name'].lower() for known in known_drugs)
        res['status'] = "✅ Known Treatment" if is_known else "🆕 Potential Discovery"
        final_output.append(res)

    return final_output

print("✅ Result processing function defined")

✅ Result processing function defined


## Section 7: Execute Complete Pipeline

⚠️ **NOTE:** The first execution will:
1. Load the DeepPurpose MPNN_CNN model (~30-60 seconds)
2. Download ADME dataset from TDC (~2-5 minutes)
3. Run virtual screening (~5-30 minutes depending on number of drugs)

This is a CPU-only setup, so please be patient!

In [14]:
import time

# Configuration
user_disease = "Type 2 Diabetes"
max_drugs = 500  # Limit to 500 drugs for faster execution (use None for all ~600)

try:
    # 1. Initialize AI Model
    print("=" * 70)
    print("🔬 DRUG REPURPOSING PIPELINE - EXECUTION START")
    print("=" * 70)
    
    start_time = time.time()
    
    print("\n[1/5] Loading Pre-trained AI Model (MPNN_CNN_BindingDB)...")
    model = models.model_pretrained(model='MPNN_CNN_BindingDB')
    print("✅ Model loaded successfully")
    
    # 2. Disease-to-Targets
    print(f"\n[2/5] Identifying disease targets for: {user_disease}")
    targets = get_disease_targets(user_disease)
    
    if not targets:
        print(f"❌ Could not find disease: {user_disease}")
    else:
        # 3. Fetch Protein Sequences
        print(f"\n[3/5] Fetching protein sequences...")
        targets_with_seqs = get_protein_sequences(targets)
        
        # 4. Load Drug Library
        print(f"\n[4/5] Loading drug library (limit: {max_drugs} drugs)...")
        print("   ⏳ This may take 2-5 minutes for first download...")
        drugs = load_drug_library(limit=max_drugs)
        
        # 5. Run the AI Virtual Screening
        print(f"\n[5/5] Running AI virtual screening...")
        print("   ⏳ This may take 5-30 minutes depending on drug count...")
        raw_results = run_virtual_screening(drugs, targets_with_seqs, model)
        
        # 6. Process and Filter Results
        print(f"\nProcessing results...")
        diabetes_meds = ["Metformin", "Sitagliptin", "Insulin", "Glipizide", "GLP1R"]
        final_candidates = process_final_results(raw_results, diabetes_meds)
        
        # 7. Display Results
        elapsed = time.time() - start_time
        print(f"\n{'=' * 80}")
        print(f"✅ SCREENING COMPLETE in {elapsed:.1f} seconds")
        print(f"{'=' * 80}")
        print(f"\n{'Drug Name':<15} | {'Target':<12} | {'Score':<8} | {'Status'}")
        print("-" * 80)
        for res in final_candidates[:15]:
            print(f"{res['drug_name']:<15} | {res['target_symbol']:<12} | {res['score']:<8.4f} | {res['status']}")
        
        print(f"\n📊 Summary:")
        print(f"   Total candidates: {len(final_candidates)}")
        print(f"   Top scores: {final_candidates[0]['score']:.4f} to {final_candidates[-1]['score']:.4f}")
        print(f"   Known treatments: {sum(1 for r in final_candidates if '✅' in r['status'])}")
        print(f"   Novel candidates: {sum(1 for r in final_candidates if '🆕' in r['status'])}")
        
except Exception as e:
    print(f"\n❌ ERROR: {str(e)}")
    import traceback
    traceback.print_exc()

🔬 DRUG REPURPOSING PIPELINE - EXECUTION START

[1/5] Loading Pre-trained AI Model (MPNN_CNN_BindingDB)...
pretrained model Successfully Downloaded...
✅ Model loaded successfully

[2/5] Identifying disease targets for: Type 2 Diabetes
✅ Target Disease Identified: type 2 diabetes mellitus (MONDO_0005148)

[3/5] Fetching protein sequences...

🧬 Fetching Amino Acid Sequences from UniProt...
   ✅ Successfully fetched sequence for: KCNJ11
   ✅ Successfully fetched sequence for: ABCC8
   ✅ Successfully fetched sequence for: GCK
   ✅ Successfully fetched sequence for: PPARG
   ✅ Successfully fetched sequence for: INSR
   ✅ Successfully fetched sequence for: HNF1B
   ✅ Successfully fetched sequence for: HNF1A
   ✅ Successfully fetched sequence for: HNF4A
   ✅ Successfully fetched sequence for: WFS1


Downloading...


   ✅ Successfully fetched sequence for: GLP1R

✅ Successfully fetched sequences for 10/10 targets

[4/5] Loading drug library (limit: 500 drugs)...
   ⏳ This may take 2-5 minutes for first download...

💊 Loading FDA-Approved Drug Library from TDC (Half_Life_Obach)...


100%|██████████| 53.6k/53.6k [00:00<00:00, 249kiB/s] 
Loading...
Done!


   Downloaded 667 total drugs from TDC
✅ Loaded 500 valid drugs ready for screening.
   Sample SMILES: CCN1CC(CCN2CCOCC2)C(c2ccccc2)(c2ccccc2)C1=O...

[5/5] Running AI virtual screening...
   ⏳ This may take 5-30 minutes depending on drug count...

🚀 Preparing data for AI Prediction (Virtual Screening)...
   Total drug-target pairs: 5000
   Encoding compounds (MPNN) and proteins (CNN)...
Drug Target Interaction Prediction Mode...
in total: 5000 drug-target pairs
encoding drug...
unique drugs: 499
encoding protein...
unique target sequence: 10
splitting dataset...
do not do train/test split on the data for already splitted data
🧠 AI is predicting Binding Affinities...
predicting...
✅ Completed 5000 predictions

Processing results...

✅ SCREENING COMPLETE in 36.5 seconds

Drug Name       | Target       | Score    | Status
--------------------------------------------------------------------------------
Drug_156        | PPARG        | 45.3584  | 🆕 Potential Discovery
Drug_88         | WFS

## 📝 Troubleshooting & Tips

### If imports fail:
```bash
# On Google Colab:
!pip install DeepPurpose
!pip install git+https://github.com/Alantic/TDC.git
!pip install rdkit
!pip install git+https://github.com/bp-kelley/descriptastorus
!pip install pandas-flavor

# On local machine, run from terminal:
pip install DeepPurpose
pip install git+https://github.com/Alantic/TDC.git
pip install rdkit
pip install git+https://github.com/bp-kelley/descriptastorus
pip install pandas-flavor
```

### Performance Notes:
- **CPU (no GPU):** 5-30 minutes per screening
- **GPU (NVIDIA):** <1 minute per screening
- **Datasets:** TDC downloads cached locally after first run

### Extend the pipeline:
- Change `user_disease` to screen any disease
- Increase `max_drugs` for more comprehensive screening
- Modify `diabetes_meds` list for different known treatments

### Output Interpretation:
- **Score 0.7+:** High binding affinity (strong prediction)
- **Score 0.5-0.7:** Moderate binding affinity (worth investigating)
- **Score <0.5:** Low binding affinity (unlikely to work)

---
**Citation:** This pipeline combines:
- DeepPurpose (Huang & Carbone, 2020)
- TDC - Therapeutic Data Commons
- Open Targets Platform
- UniProt Database